# Qwen2.5-1.5B task-extraction ORPO

Fine-tunes the model Xayra ships for on-device to-do extraction.

## Why this notebook is ORPO, not plain SFT

A 103-case evaluation harness put the prompt-only baseline at **83.5%**, with two failures three rounds of prompt work could not move: third-party attribution (0/8) and zero-task refusal (11/17, i.e. the model invented tasks from purely descriptive notes).

Two plain-SFT attempts at fixing this both broke in opposite directions, using the same lever — the ratio of positive-to-refusal training examples:

| Run | Refusal share | Fixed | Broke |
|---|---|---|---|
| 2 | 50% | third-party 0/8→8/8, refusal 11/17→17/17 | date-resolution 20/20→6/20, recurrence 8/8→1/8 |
| 3 | 15% | date-resolution 6/20→19/20, recurrence 1/8→8/8 | third-party 8/8→0/8, refusal 17/17→6/17 |

Both runs showed the same mechanism: plain SFT only ever teaches "this completion is correct," never "this completion is wrong." With no negative signal, the only lever for avoiding one failure mode is more examples of the opposite behaviour — which just pushes the model into the other failure mode once that behaviour dominates. There is no ratio that pins both ends of this seesaw down at once.

**ORPO** (Odds Ratio Preference Optimization) trains on `(prompt, chosen, rejected)` triples and directly penalises the wrong completion for a given note, not just rewards the right one. A refusal note's `rejected` side is a hallucinated task (the exact "Garden" / "Plumber" / "Look at the roof" shape Run 3 produced on-device); a task note's `rejected` side is bare `"[]"` (Run 2's failure). Both lessons live in one file, scored against each other, rather than in two separately-tuned datasets.

## Runtime

Free Colab **T4**. Set `Runtime -> Change runtime type -> T4 GPU` before running. Roughly 10-20 minutes end to end for 350 pairs over 2 epochs.

## 1. Dependencies

Unsloth pins compatible `trl`/`peft`/`xformers` builds itself; installing them loose alongside it is the usual cause of a broken Colab session. `trl` already ships `ORPOTrainer`/`ORPOConfig` in any release from the last two years, so no extra package or version pin is needed for the ORPO switch.

In [ ]:
%%capture
import torch

major, _ = torch.cuda.get_device_capability()

!pip install -q --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

if major >= 8:
    # Ampere and newer (A100, L4): flash-attn and bf16 are available.
    !pip install -q --no-deps packaging ninja einops flash-attn xformers trl peft accelerate bitsandbytes
else:
    # T4 is Turing (sm75): no flash-attn, no bf16. Unsloth falls back to fp16.
    !pip install -q --no-deps xformers trl peft accelerate bitsandbytes

In [ ]:
import torch

print("GPU        :", torch.cuda.get_device_name(0))
print("Capability :", torch.cuda.get_device_capability())
print("VRAM (GB)  :", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
print("bf16       :", torch.cuda.is_bf16_supported())

## 2. Dataset

Upload `orpo_qwen_task_extraction.jsonl`, produced by `scripts/dataset/generate_sft.py` (v4). Each row is already `{"prompt", "chosen", "rejected", "kind"}` — `prompt` ends exactly where generation begins (`<|im_start|>assistant\n`), and `chosen`/`rejected` are the two completions ORPO scores against each other for that prompt.

RAG samples are NOT in this file — `ORPOTrainer` requires every row to carry `prompt`/`chosen`/`rejected`, and no rejected-answer design for RAG was part of this pivot. This file is extraction-only.

In [ ]:
from google.colab import files

uploaded = files.upload()  # select orpo_qwen_task_extraction.jsonl
DATASET_PATH = next(iter(uploaded))
print("Using:", DATASET_PATH)

In [ ]:
import json
from collections import Counter
from datasets import Dataset

rows = [json.loads(line) for line in open(DATASET_PATH, encoding="utf-8") if line.strip()]

# Every row must actually carry the three ORPOTrainer needs -- a file that
# silently lost one (the same class of schema drift that broke this
# notebook's SFT-era validation cell twice already) should fail HERE, not
# after a training run.
required_fields = {"prompt", "chosen", "rejected"}
for i, r in enumerate(rows):
    missing = required_fields - set(r)
    assert not missing, f"row {i} missing {missing}"

counts = Counter(r["kind"] for r in rows)
total = len(rows)
print(f"pairs            : {total}")
for kind in sorted(counts):
    print(f"  {kind:24s} {counts[kind]:4d}  ({counts[kind] / total:.0%})")

# ORPO-specific checks: the whole point of this file is that BOTH failure
# directions are represented. A degenerate row (chosen == rejected) would
# train the model to be indifferent between right and wrong for that note.
degenerate = sum(1 for r in rows if r["chosen"] == r["rejected"]) 
rejected_empty = sum(1 for r in rows if r["rejected"].startswith("[]"))
chosen_empty = sum(1 for r in rows if r["chosen"].startswith("[]"))
print()
print(f"rejected == '[]' (over-refusal lesson) : {rejected_empty}")
print(f"chosen == '[]' (over-extraction lesson) : {chosen_empty}")
print(f"degenerate (chosen == rejected)         : {degenerate}  (must be 0)")

assert degenerate == 0, "some rows have identical chosen/rejected -- regenerate the file"
assert rejected_empty > 0 and chosen_empty > 0, "missing one of the two preference directions"
required_kinds = {"extraction_positive", "extraction_refusal", "extraction_contrastive"}
missing_kinds = required_kinds - set(counts)
assert not missing_kinds, f"missing kinds: {missing_kinds}"
assert counts["extraction_contrastive"] >= 75, (
    f"only {counts['extraction_contrastive']} contrastive pairs, need >=75"
)

dataset = Dataset.from_list([
    {"prompt": r["prompt"], "chosen": r["chosen"], "rejected": r["rejected"]} for r in rows
])
print()
print("--- prompt ---")
print(dataset[0]["prompt"])
print("--- chosen ---")
print(dataset[0]["chosen"])
print("--- rejected ---")
print(dataset[0]["rejected"])

## 3. Model — QLoRA via Unsloth

Unaffected by the SFT->ORPO switch: `FastLanguageModel.from_pretrained`/`get_peft_model` load and adapt the base model the same way regardless of which trainer updates the LoRA weights afterward. `r=16` with `lora_alpha=16` (1:1) stays deliberate for a corrective fine-tune — the goal is suppressing specific behaviours, not teaching a new domain.

In [ ]:
from unsloth import FastLanguageModel

MAX_SEQ_LENGTH = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-1.5B-Instruct",
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,      # None lets Unsloth pick fp16 on T4, bf16 on Ampere+
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,          # 0 is Unsloth's optimised path
    bias="none",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    use_gradient_checkpointing="unsloth",
    random_state=20260921,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"trainable params: {trainable:,}")

## 4. Train — ORPO

`beta=0.1`, `learning_rate=5e-5`, 2 epochs. `PatchDPOTrainer()` — not a separate `PatchORPOTrainer`, which does not exist anywhere in Unsloth — patches the memory-efficient preference-loss kernels into **both** `DPOTrainer` and `ORPOTrainer`; it is called once, after the dataset is ready, right before the trainer is constructed. Confirmed directly against Unsloth's own reference ORPO notebook before writing this cell, rather than assumed from the DPO pattern alone.

`max_length=1024` / `max_prompt_length=512` split the sequence budget between the shared prompt and the two completions; `max_completion_length` is derived as `max_length - max_prompt_length` (512) rather than left at whatever ORPOConfig defaults to, since the three are meant to be consistent with each other.

In [ ]:
from unsloth import PatchDPOTrainer

PatchDPOTrainer()  # patches both DPOTrainer and ORPOTrainer -- there is no separate PatchORPOTrainer

from trl import ORPOConfig, ORPOTrainer

MAX_PROMPT_LENGTH = 512
MAX_LENGTH = 1024

orpo_trainer = ORPOTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=ORPOConfig(
        max_length=MAX_LENGTH,
        max_prompt_length=MAX_PROMPT_LENGTH,
        max_completion_length=MAX_LENGTH - MAX_PROMPT_LENGTH,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,   # effective batch size 8
        num_train_epochs=2,
        learning_rate=5e-5,
        beta=0.1,
        warmup_ratio=0.1,
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        seed=20260921,
        output_dir="outputs",
        report_to="none",
    ),
)

stats = orpo_trainer.train()
print(stats)

## 5. Smoke test before export

Four prompts covering both target failures and both behaviours that must **not** regress. Worth thirty seconds here rather than discovering a collapsed model after a 1GB download — but a clean pass here is NOT proof the fix worked: this exact smoke test passed 4/4 on Run 2's checkpoint, which then failed the full harness. Section 6 below is the real pre-export gate.

Expected: `[]`, `[]`, a single task, two tasks.

In [ ]:
SYSTEM_PROMPT = (
    "You are an executive task extraction assistant. Extract actionable user "
    "tasks into the requested JSON schema. If no tasks exist for the user, "
    "return []."
)

PROBES = [
    ("The builder is coming Tuesday to look at the roof.", "[] -- third party"),
    ("Feeling much better today than yesterday.", "[] -- observation"),
    ("Remind me to call the dentist tomorrow.", "one task"),
    ("Book the flights and renew the travel insurance.", "two tasks"),
]

FastLanguageModel.for_inference(model)

def generate(note):
    prompt = (
        f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
        f"<|im_start|>user\n{note}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
    out = model.generate(**inputs, max_new_tokens=128, temperature=0.0, do_sample=False)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

for note, expectation in PROBES:
    answer = generate(note)
    print(f"{note}\n  expect: {expectation}\n  got   : {answer}\n")

## 6. Automated 20-case validation sweep

The 4-probe smoke test above is not sufficient on its own — proven directly by this project's own history, not a hypothetical caveat. This sweep is a wider, still-quick (well under a minute) gate before spending 10-15 minutes on GGUF export: 8 third-party, 4 pure refusal, 4 dated tasks, 4 contrastive (task + distractor together, the specific mechanism this whole pivot targets). None of these sentences exist in the training file.

**This is a gate, not the verdict.** Even a clean 20/20 here does not replace running the exported GGUF through the real 103-case harness afterward — 20 hand-picked cases could not have caught, for instance, Run 2's specific date-resolution collapse, since none of the four original smoke-test probes touched dates at all. Treat a pass here as "worth spending the export time," not as "done."

In [ ]:
import json as _json

# (note, kind, check) -- `check` takes the raw generated text and returns
# (passed: bool, detail: str). Kept intentionally simpler than the real
# TypeScript harness (scripts/eval/scoring.ts) -- this only has to decide
# "is this checkpoint worth exporting," not produce a publishable score.

def _is_empty(text):
    try:
        return _json.loads(text) == []
    except Exception:
        return False

def _has_a_task(text):
    try:
        parsed = _json.loads(text)
        return isinstance(parsed, list) and len(parsed) > 0
    except Exception:
        return False

def _has_task_without_leaking(forbidden):
    def _check(text):
        if not _has_a_task(text):
            return False
        lower = text.lower()
        return not any(word.lower() in lower for word in forbidden)
    return _check

VALIDATION_CASES = [
    # -- third-party (8): must all return [] --
    ("The plumber is coming Thursday to fix the boiler.", "third_party", _is_empty),
    ("My sister is moving house next month.", "third_party", _is_empty),
    ("The council is collecting green waste next week.", "third_party", _is_empty),
    ("A courier is delivering the parcel tomorrow afternoon.", "third_party", _is_empty),
    ("The cleaner comes every second Friday.", "third_party", _is_empty),
    ("My colleague is presenting at the conference in March.", "third_party", _is_empty),
    ("The gas company is reading the meter on Friday.", "third_party", _is_empty),
    ("The neighbours are having their driveway resurfaced.", "third_party", _is_empty),
    # -- pure refusal / observation (4): must all return [] --
    ("The new bakery bread is better than the supermarket one.", "refusal", _is_empty),
    ("Traffic was much lighter than usual this morning.", "refusal", _is_empty),
    ("That podcast episode on sleep was surprisingly good.", "refusal", _is_empty),
    ("The old bike is holding up better than expected.", "refusal", _is_empty),
    # -- dated tasks (4): must extract a task, must not be empty --
    ("Renew the parking permit by Friday.", "date", _has_a_task),
    ("Call the accountant next Tuesday about the invoice.", "date", _has_a_task),
    ("Pay the storage fee on the 5th of every month.", "date", _has_a_task),
    ("Collect the dry cleaning this Saturday.", "date", _has_a_task),
    # -- contrastive (4): must extract ONLY the real task, distractor must not leak --
    ("The electrician is coming Tuesday to check the wiring. Buy lightbulbs tomorrow.",
     "contrastive", _has_task_without_leaking(["electrician", "wiring"])),
    ("My cousin is moving to Berlin next month. Cancel the newspaper subscription.",
     "contrastive", _has_task_without_leaking(["cousin", "Berlin"])),
    ("The new cafe down the street is fantastic. Water the plants every Tuesday.",
     "contrastive", _has_task_without_leaking(["cafe"])),
    ("The neighbours are getting their roof redone this week. Book the dentist for next Wednesday.",
     "contrastive", _has_task_without_leaking(["neighbours", "roof"])),
]

assert len(VALIDATION_CASES) == 20

results = []
for note, category, check in VALIDATION_CASES:
    answer = generate(note)
    passed = check(answer)
    results.append((category, passed))
    mark = "PASS" if passed else "FAIL"
    print(f"[{mark}] ({category}) {note}\n       -> {answer}\n")

print("=" * 60)
by_category = {}
for category, passed in results:
    total, ok = by_category.get(category, (0, 0))
    by_category[category] = (total + 1, ok + (1 if passed else 0))
for category, (total, ok) in sorted(by_category.items()):
    print(f"{category:14s} {ok}/{total}")
overall = sum(1 for _, p in results if p)
print(f"{'TOTAL':14s} {overall}/{len(results)}")
if overall < len(results):
    print("\nNot a clean sweep -- worth reading the failures above before spending the export step.")
else:
    print("\nClean sweep. Proceed to export -- the real verdict is still the 103-case harness afterward.")

## 7. Export to GGUF (q4_k_m)

Matches the quantization the app already ships. Unaffected by ORPO vs SFT — this exports the merged LoRA weights regardless of which trainer produced them. This step builds llama.cpp from source the first time and is the slowest cell in the notebook — 10-15 minutes is normal.

In [ ]:
model.save_pretrained_gguf(
    "qwen-task-extractor",
    tokenizer,
    quantization_method="q4_k_m",
)

!ls -lh qwen-task-extractor/*.gguf

In [ ]:
import glob
from google.colab import files

gguf = glob.glob("qwen-task-extractor/*.gguf")[0]
print("Downloading", gguf)
files.download(gguf)

## 8. Next steps, on your machine

```bash
# 1. Put the exported model where the harness looks
mv ~/Downloads/*.gguf models/qwen-task-extractor-q4_k_m.gguf

# 2. Score it against the frozen 103-case corpus, with the minimal prompt
XAYRA_EXTRACTION_PROMPT=minimal \
  npm run eval -- --model models/qwen-task-extractor-q4_k_m.gguf \
                  --corpus scripts/eval/corpus-full.jsonl
```

The bar to beat is **83.5% (86/103)** — the prompt-only stock baseline, still undefeated by either SFT run. Watch specifically:

- `third-party` and `refusal` — Run 2 got these to 100% and Run 3 lost them entirely
- `date-resolution` and `recurrence` — Run 3 got these to ~100% and Run 2 lost them entirely

ORPO exists specifically to hold BOTH of those at once, which neither plain-SFT run could do. If this run does that, it is the first checkpoint worth shipping. If it does not, that is real signal about where the remaining generalization gap is — not a reason to keep tuning the same lever a third time.